# walkthrough: from chain rule to backprop

this notebook shows how the `Value` engine works, step by step. the idea is
that every number remembers how it was made, so we can walk backwards and
fill in the gradients with the chain rule. run the setup cell first on colab.

In [ ]:
# colab setup (skip if running locally)
!git clone https://github.com/Devansh-hello/tiny-autograd.git
%cd tiny-autograd
!pip install -e . -q

In [ ]:
from tinygrad_scratch.engine import Value

a = Value(2.0)
b = Value(-3.0)
c = a * b + a.tanh()
c.backward()
print('c =', c.data)
print('dc/da =', a.grad)
print('dc/db =', b.grad)

let's check `dc/da` by hand. c = a*b + tanh(a), so dc/da = b + (1 - tanh(a)^2).
we can compare that with a finite difference to be sure.

In [ ]:
import math

def f(x):
    return x * -3.0 + math.tanh(x)

eps = 1e-6
numeric = (f(2.0 + eps) - f(2.0 - eps)) / (2 * eps)
print('analytic', a.grad)
print('numeric ', numeric)

they match. the same backward pass trains a real classifier: below is the
two-moons setup from `tests/test_train_moons.py` (a 40-point subset so it runs
in under a minute on the scalar engine).

In [ ]:
import random
import numpy as np
from sklearn.datasets import make_moons
from tinygrad_scratch.nn import MLP, mse_loss
from tinygrad_scratch.optim import Adam

random.seed(0); np.random.seed(0)
X, y = make_moons(n_samples=40, noise=0.2, random_state=0)
xs, targets = X.tolist(), [1.0 if t == 1 else -1.0 for t in y]
model = MLP(2, [8, 8, 1])
opt = Adam(model.parameters(), lr=0.05)
for epoch in range(101):
    preds = [model(x) for x in xs]
    loss = mse_loss(preds, targets)
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 25 == 0:
        acc = sum(((m.data > 0) == (t > 0)) for m, t in zip(preds, targets)) / len(xs)
        print(f"epoch {epoch:3d}  loss {loss.data:.4f}  acc {acc:.3f}")

the full 200-point, 120-epoch run (`scripts/train_moons.py`) reaches 0.985
accuracy, and `scripts/benchmark_parity.py` checks every operator's gradient
against PyTorch float64 autograd over 500 random expressions (worst max
difference ~3.5e-15).